# PIPELINE DE PREPRODUCCIÓN: RIESGO DE CRÉDITO

## IMPORTAR PAQUETES

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

#RUTAS
# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ----------USO DE UTILS.PY------------
import sys
from pathlib import Path
# Subir de 03_notebooks a 03_RIESGOS
ROOT = Path.cwd().parent
# Añadir 04_scripts a las rutas de búsqueda
sys.path.insert(0, str(ROOT / "04_scripts"))
#---------------------------------------

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## IMPORTACIÓN DE DATOS

In [18]:
# ── Lectura del CSV original y separación de la reserva de 01 ─────────
# En 01_Importacion_Datos: datos = pd.read_csv(ORIGINALES / 'prestamos.csv'),
# val = datos.sample(frac=0.3, random_state=42) y trabajo = datos.loc[~datos.index.isin(val.index)]
datos = pd.read_csv(ORIGINALES / 'prestamos.csv')

# La reserva se lee de 01, no se vuelve a muestrear: así es exactamente la misma
val = pd.read_pickle(VALIDACION / 'prestamos_val.pkl')

# Desde aquí todo el notebook trabaja solo con el 70 % restante
df = datos.loc[~datos.index.isin(val.index)].copy()
print(datos.shape, val.shape, df.shape)


(200000, 25) (60000, 25) (140000, 25)


## CALIDAD DE DATOS

In [19]:
# ── Columnas y registros descartados ──────────────────────────────────
df.reset_index(drop=True, inplace=True)
df.drop(columns='id_prestamo', inplace=True)

# ── Separación por tipo de dato ───────────────────────────────────────
cat = df.select_dtypes(include='string')
num = df.select_dtypes(include='number')

# ── Categóricas: textos descartados e imputación de la antigüedad ─────
cat.drop(columns=['empleo', 'descripcion'], inplace=True)
cat['antigüedad_empleo'] = cat['antigüedad_empleo'].fillna('Desconocido')

# ── Numéricas: nulos a cero y recuentos a entero ──────────────────────
num.fillna(0, inplace=True)
num['num_hipotecas'] = num['num_hipotecas'].astype(int)
num['num_lineas_credito'] = num['num_lineas_credito'].astype(int)
num['num_cancelaciones_12meses'] = num['num_cancelaciones_12meses'].astype(int)
num['num_derogatorios'] = num['num_derogatorios'].astype(int)
num['num_meses_desde_ult_retraso'] = num['num_meses_desde_ult_retraso'].astype(int)

# ── Ingresos fuera de rango ───────────────────────────────────────────
a_eliminar = num.loc[num['ingresos'] > 400000].index.values
cat = cat[~cat.index.isin(a_eliminar)]
num = num[~num.index.isin(a_eliminar)]
print(cat.shape, num.shape)

(139602, 7) (139602, 15)


## EDA Y TRANSFORMACIONES

In [20]:
# ── Nombre de la antigüedad sin tilde ─────────────────────────────────
cat = cat.rename(columns={'antigüedad_empleo': 'antiguedad_empleo'})

# ── Recorte de atípicos ───────────────────────────────────────────────
num['dti'] = np.clip(num['dti'], 0, 100)
num['num_hipotecas'] = np.clip(num['num_hipotecas'], 0, 7)
num['porc_uso_revolving'] = np.clip(num['porc_uso_revolving'], 0, 100)

## INGENIERÍA DE VARIABLES

In [21]:
# ── Variable objetivo PD ──────────────────────────────────────────────
lista_estados = ['Charged Off', 'Does not meet the credit policy. Status:Charged Off', 'Default']
cat['target_pd'] = np.where(cat['estado'].isin(lista_estados), 1, 0)
cat.drop('estado', axis=1, inplace=True)   # en producción no se conoce

# ── Variables objetivo EAD y LGD ──────────────────────────────────────
num['pendiente'] = num['principal'] - num['imp_amortizado']
num['target_ead'] = num['pendiente'] / num['principal']
num['target_lgd'] = 1 - (num['imp_recuperado'] / num['pendiente'])
num['target_lgd'] = num['target_lgd'].fillna(0)
num['target_ead'] = num['target_ead'].clip(0, 1)
num['target_lgd'] = num['target_lgd'].clip(0, 1)

# ── Categorías con muy pocos casos ────────────────────────────────────
cat['vivienda'] = cat['vivienda'].replace(['ANY', 'NONE', 'OTHER'], 'MORTGAGE')
cat['finalidad'] = cat['finalidad'].replace(['wedding', 'educational', 'renewable_energy'], 'other')

# ── Columna descartada ────────────────────────────────────────────────
num.drop(columns='num_meses_desde_ult_retraso', inplace=True)

# ── One Hot Encoding ──────────────────────────────────────────────────
from sklearn.preprocessing import Binarizer, MinMaxScaler, OneHotEncoder, OrdinalEncoder

vars_to_onehot = ['ingresos_verificados', 'vivienda', 'finalidad', 'num_cuotas']
onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_ohe = onehot.fit_transform(cat[vars_to_onehot])
cat_ohe = pd.DataFrame(cat_ohe, columns=onehot.get_feature_names_out(vars_to_onehot))

# ── Ordinal Encoding ──────────────────────────────────────────────────
# Orden de menor a mayor antigüedad. Los textos deben ser IGUALES a los de los datos.
orden_antiguedad_empleo = ['Desconocido', '< 1 year', '1 year', '2 years', '3 years',
                           '4 years', '5 years', '6 years', '7 years', '8 years',
                           '9 years', '10+ years']
orden_rating = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
vars_to_ordinal = ['antiguedad_empleo', 'rating']
cat_oe = OrdinalEncoder(categories=[orden_antiguedad_empleo, orden_rating], handle_unknown='use_encoded_value',
                        unknown_value=12)
cat_oe = cat_oe.fit_transform(cat[vars_to_ordinal])
cat_oe = pd.DataFrame(cat_oe, columns=vars_to_ordinal)

# ── Binarización de num_derogatorios ──────────────────────────────────
num_bin = Binarizer(threshold=0).fit_transform(num[['num_derogatorios']])
num_bin = pd.DataFrame(num_bin, columns=['num_derogatorios'])

# ── Escalado Min Max ──────────────────────────────────────────────────
# Se concatenan las categóricas ordinales con las numéricas a escalar: las que llevan
# one hot ya están entre 0 y 1.
num_escalar = num[['ingresos',
                   'dti',
                   'num_lineas_credito',
                   'porc_uso_revolving',
                   'principal',
                   'tipo_interes',
                   'imp_cuota']].reset_index(drop=True)
df_res = pd.concat([cat_oe, num_escalar], axis=1)
var_mms = df_res.columns.tolist()
scaler = MinMaxScaler()
num_escalar_scaled = scaler.fit_transform(df_res[var_mms])
var_mms = [col + '_scaled' for col in var_mms]
df_mms = pd.DataFrame(num_escalar_scaled, columns=var_mms)

# ── Tablón PD ─────────────────────────────────────────────────────────
id_cliente = num[['id_cliente']].reset_index(drop=True)
variables = pd.concat(
    [
        cat_ohe.reset_index(drop=True),
        df_mms.reset_index(drop=True),
        num_bin.reset_index(drop=True)
    ],
    axis=1
)
y_pd = cat['target_pd'].reset_index(drop=True).rename('target_pd')
assert len(id_cliente) == len(variables) == len(y_pd)
tabla_pd = pd.concat([id_cliente, variables, y_pd], axis=1)
tabla_pd.set_index('id_cliente', inplace=True)
assert tabla_pd.index.name == 'id_cliente'
assert tabla_pd.index.isnull().sum() == 0

# ── Tablones EAD y LGD, solo con préstamos en default ─────────────────
target_pd = cat['target_pd'].reset_index(drop=True).rename('target_pd')
y_ead = num['target_ead'].reset_index(drop=True).rename('target_ead')
y_lgd = num['target_lgd'].reset_index(drop=True).rename('target_lgd')
assert len(id_cliente) == len(variables) == len(target_pd) == len(y_ead) == len(y_lgd)

tabla_ead = pd.concat([id_cliente, variables, target_pd, y_ead], axis=1)
tabla_ead = tabla_ead.loc[tabla_ead['target_pd'] == 1]
tabla_ead.set_index('id_cliente', inplace=True)

tabla_lgd = pd.concat([id_cliente, variables, target_pd, y_lgd], axis=1)
tabla_lgd = tabla_lgd.loc[tabla_lgd['target_pd'] == 1].copy()
tabla_lgd.set_index('id_cliente', inplace=True)

print(tabla_pd.shape, tabla_ead.shape, tabla_lgd.shape)

(139602, 30) (16549, 31) (16549, 31)


## MODELO PD: PROBABILITY OF DEFAULT

In [22]:
# ── Modelo PD: regresión logística con Lasso ──────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

x = tabla_pd.drop(columns='target_pd')
y = tabla_pd['target_pd']

# 70 % entrenamiento y 30 % validación, estratificado por el objetivo (como en 07)
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

modelo_pd = LogisticRegression(solver='saga', l1_ratio=1, C=0.05, max_iter=5000)
modelo_pd.fit(x_train, y_train)

pred_pd = modelo_pd.predict_proba(x_val)[:, 1]
print('AUC validación:', roc_auc_score(y_val, pred_pd))


AUC validación: 0.6949996873771401


## MODELO EAD: EXPOSURE AT DEFAULT

In [23]:
# ── Modelo EAD: HistGradientBoosting ──────────────────────────────────
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

x_ead = tabla_ead.drop(columns=['target_pd', 'target_ead'])
y_ead = tabla_ead['target_ead']

# 70 % entrenamiento y 30 % validación (como en 08)
x_train, x_val, y_train, y_val = train_test_split(x_ead, y_ead, test_size=0.3, random_state=42)

modelo_ead = HistGradientBoostingRegressor(learning_rate=0.05, l2_regularization=0.25, max_depth=5, max_iter=200,
                                           min_samples_leaf=60, scoring='neg_mean_absolute_error')
modelo_ead.fit(x_train, y_train)

# ── Recorte de la predicción a [0, 1] ─────────────────────────────────
pred_ead = modelo_ead.predict(x_val)
pred_ead = np.where(pred_ead < 0, 0, pred_ead)
pred_ead = np.where(pred_ead > 1, 1, pred_ead)
print('MAE validación:', mean_absolute_error(y_val, pred_ead))


MAE validación: 0.15777533855848583


## MODELO LGD: LOSS GIVEN DEFAULT

In [24]:
# ── Modelo LGD: HistGradientBoosting ──────────────────────────────────
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

x_lgd = tabla_lgd.drop(columns=['target_pd', 'target_lgd'])
y_lgd = tabla_lgd['target_lgd']

# 70 % entrenamiento y 30 % validación (como en 09)
x_train, x_val, y_train, y_val = train_test_split(x_lgd, y_lgd, test_size=0.3, random_state=42)

modelo_lgd = HistGradientBoostingRegressor(learning_rate=0.01, l2_regularization=0, max_depth=5, max_iter=50,
                                           min_samples_leaf=60, scoring='neg_mean_absolute_error')
modelo_lgd.fit(x_train, y_train)

# ── Recorte de la predicción a [0, 1] ─────────────────────────────────
pred_lgd = modelo_lgd.predict(x_val)
pred_lgd = np.where(pred_lgd < 0, 0, pred_lgd)
pred_lgd = np.where(pred_lgd > 1, 1, pred_lgd)
print('MAE validación:', mean_absolute_error(y_val, pred_lgd))


MAE validación: 0.08866732514454177
